In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [26]:
url = "https://www.stevens.edu/hass/hass-faculty"

In [27]:
response = requests.get(url)

In [28]:
soup = BeautifulSoup(response.content, "html.parser")

In [29]:
soup

<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"/><meta content="width=device-width" name="viewport"/><meta content="summary_large_image" name="twitter:card"/><meta content="@FollowStevens" name="twitter:site"/><meta content="@FollowStevens" name="twitter:creator"/><meta content="website" property="og:type"/><meta content="en_US" property="og:locale"/><meta content="Stevens Institute of Technology" property="og:site_name"/><title>Faculty of the School of Humanities, Arts and Social Sciences | Stevens Institute of Technology</title><meta content="index,follow" name="robots"/><meta content="Faculty of the School of Humanities, Arts and Social Sciences" property="og:title"/><meta content="https://www.stevens.edu/hass/hass-faculty" property="og:url"/><link href="https://www.stevens.edu/hass/hass-faculty" rel="canonical"/><meta content="13" name="next-head-count"/><link href="/favicon/apple-touch-icon.png" rel="apple-touch-icon" sizes="180x180"/><link href="/favicon/favicon-32x32

In [30]:
faculty_data = []


In [31]:
import re
cards = soup.find_all("div", class_=re.compile(r"faculty-card_c--faculty-card__"))


In [32]:
cards[1]

<div class="faculty-card_c--faculty-card__y49jx"><div class="faculty-card_image-wrapper__X2pYq"><a aria-label="Link to Michelle Burke" href="/profile/mburke3"><span style="box-sizing:border-box;display:block;overflow:hidden;width:initial;height:initial;background:none;opacity:1;border:0;margin:0;padding:0;position:relative"><span style="box-sizing:border-box;display:block;width:initial;height:initial;background:none;opacity:1;border:0;margin:0;padding:0;padding-top:100%"></span><noscript><img alt="Image of Michelle Burke" data-nimg="responsive" decoding="async" loading="lazy" sizes="100vw" src="/_next/image?url=https%3A%2F%2Fimages.ctfassets.net%2Fmviowpldu823%2F1p6KcdOMBVvw5dvsOUHeqw%2Fd483cfa1acfec3f83ed17387f165b644%2Fmburke3.jpg%3Fw%3D300%26h%3D300%26f%3Dface%26q%3D80%26fit%3Dfill&amp;w=2400&amp;q=80" srcset="/_next/image?url=https%3A%2F%2Fimages.ctfassets.net%2Fmviowpldu823%2F1p6KcdOMBVvw5dvsOUHeqw%2Fd483cfa1acfec3f83ed17387f165b644%2Fmburke3.jpg%3Fw%3D300%26h%3D300%26f%3Dface%26q

In [33]:
base_url = "https://www.stevens.edu"
headers = {"User-Agent": "Mozilla/5.0"}
import time
for card in cards:
    name_tag = card.find("div", class_=re.compile(r"faculty-card_name-title"))
    name_link = name_tag.find("a") if name_tag else None
    name = name_link.get_text(strip=True) if name_link else ""
    profile_path = name_link.get("href", "") if name_link else ""
    profile_url = base_url + profile_path if profile_path else ""

    title_tag = card.find("h4")
    title = title_tag.get_text(strip=True) if title_tag else ""

    contact_tag = card.find("div", class_=re.compile(r"faculty-card_contact"))
    contact_info = contact_tag.get_text(separator=" ", strip=True) if contact_tag else ""

    # Scrape full text from profile page
    profile_text = ""
    if profile_url:
        try:
            time.sleep(1)  # polite delay
            profile_response = requests.get(profile_url, headers=headers, timeout=10)
            profile_soup = BeautifulSoup(profile_response.content, "html.parser")
            profile_text = profile_soup.get_text(separator=" ", strip=True)
        except Exception as e:
            print(f"Failed to fetch {profile_url}: {e}")
            profile_text = ""

    faculty_data.append({
        "Name": name,
        "Profile URL": profile_url
    })

In [34]:
faculty_data

[{'Name': 'Amber Benezra',
  'Profile URL': 'https://www.stevens.edu/profile/abenezra'},
 {'Name': 'Michelle Burke',
  'Profile URL': 'https://www.stevens.edu/profile/mburke3'},
 {'Name': 'Diana Bush',
  'Profile URL': 'https://www.stevens.edu/profile/dbush'},
 {'Name': '', 'Profile URL': ''},
 {'Name': 'Sean Cashbaugh',
  'Profile URL': 'https://www.stevens.edu/profile/scashbau'},
 {'Name': '', 'Profile URL': ''},
 {'Name': 'Betul Cihan Artun',
  'Profile URL': 'https://www.stevens.edu/profile/fcihanar'},
 {'Name': 'Virginia Conn',
  'Profile URL': 'https://www.stevens.edu/profile/vconn'},
 {'Name': 'Lindsey Cormack',
  'Profile URL': 'https://www.stevens.edu/profile/lcormack'},
 {'Name': 'Smaran Dayal',
  'Profile URL': 'https://www.stevens.edu/profile/sdayal'},
 {'Name': 'Katheryn Detwiler',
  'Profile URL': 'https://www.stevens.edu/profile/kdetwile'},
 {'Name': 'Mario Diaz de Leon',
  'Profile URL': 'https://www.stevens.edu/profile/mdiazdel'},
 {'Name': 'Aysegul Durakoglu',
  'Prof

In [35]:
from bs4 import BeautifulSoup, Tag
all_faculty_json = []

for faculty in faculty_data:
    profile_url = faculty["Profile URL"]
    time.sleep(1)  # Polite delay to avoid overwhelming the server

    try:
        response = requests.get(profile_url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"Failed to fetch {profile_url}")
            continue

        soup = BeautifulSoup(response.content, "html.parser")

        profile_info = {
            "Name": "",
            "Title": "",
            "Contact Info": {},
            "Profile URL": profile_url,
            "Sections": {}
        }

        # Extract Name
        name_tag = soup.find("h1")
        profile_info["Name"] = name_tag.get_text(strip=True) if name_tag else ""

        # Extract Title
        title_tag = soup.find("h3")
        profile_info["Title"] = title_tag.get_text(strip=True) if title_tag else ""

        # Extract Contact Info
        contact_info = {}
        contact_block = soup.find("div", class_=re.compile(r"hero-profile_profile-places"))
        if contact_block:
            # Address
            address = contact_block.find("div", class_=re.compile(r"address"))
            contact_info["Address"] = address.get_text(strip=True) if address else ""

            # Phone
            phone = contact_block.find("div", class_=re.compile(r"phone"))
            contact_info["Phone"] = phone.get_text(strip=True) if phone else ""

            # Email
            email = contact_block.find("div", class_=re.compile(r"email"))
            contact_info["Email"] = email.get_text(strip=True) if email else ""

            # Website
            website = contact_block.find("div", class_=re.compile(r"website"))
            contact_info["Website"] = website.get_text(strip=True) if website else ""
        else:
            contact_info = {"Address": "", "Phone": "", "Email": "", "Website": ""}

        profile_info["Contact Info"] = contact_info

        # Extract Sections like Education, Research, Publications, etc.
        sections = {}
        h2_tags = soup.find_all("h2")

        for header in h2_tags:
            section_title = header.get_text(strip=True)
            content_chunks = []

            for sibling in header.find_all_next():
                if sibling.name == "h2":
                    break
                if isinstance(sibling, Tag):
                    text = sibling.get_text(separator=" ", strip=True)
                    if text and text not in content_chunks:
                        content_chunks.append(text)

            section_content = " ".join(content_chunks).strip()
            if section_title and section_content:
                sections[section_title] = section_content

        profile_info["Sections"] = sections

        all_faculty_json.append(profile_info)
        print(profile_info)

    except Exception as e:
        print(f"Error scraping {profile_url}: {e}")

{'Name': 'Amber Benezra', 'Title': 'Academics', 'Contact Info': {'Address': 'Peirce 209', 'Phone': '(201) 216-8530', 'Email': '[email\xa0protected]', 'Website': 'Website'}, 'Profile URL': 'https://www.stevens.edu/profile/abenezra', 'Sections': {'Education': 'PhD (2014) New School for Social Research (Sociocultural Anthropology) MA (2007) New School for Social Research (Sociocultural Anthropology) MA (2000) New School for Social Research (Media Studies and Film) BA (1996) Carnegie Mellon University (Literary & Cultural Studies/Creative Writing) PhD (2014) New School for Social Research (Sociocultural Anthropology) MA (2007) New School for Social Research (Sociocultural Anthropology) MA (2000) New School for Social Research (Media Studies and Film) BA (1996) Carnegie Mellon University (Literary & Cultural Studies/Creative Writing) Research Amber Benezra is a sociocultural anthropologist researching how studies of the human microbiome intersect with biomedical ethics, public health/techno

In [36]:
all_faculty_json

[{'Name': 'Amber Benezra',
  'Title': 'Academics',
  'Contact Info': {'Address': 'Peirce 209',
   'Phone': '(201) 216-8530',
   'Email': '[email\xa0protected]',
   'Website': 'Website'},
  'Profile URL': 'https://www.stevens.edu/profile/abenezra',
  'Sections': {'Education': 'PhD (2014) New School for Social Research (Sociocultural Anthropology) MA (2007) New School for Social Research (Sociocultural Anthropology) MA (2000) New School for Social Research (Media Studies and Film) BA (1996) Carnegie Mellon University (Literary & Cultural Studies/Creative Writing) PhD (2014) New School for Social Research (Sociocultural Anthropology) MA (2007) New School for Social Research (Sociocultural Anthropology) MA (2000) New School for Social Research (Media Studies and Film) BA (1996) Carnegie Mellon University (Literary & Cultural Studies/Creative Writing) Research Amber Benezra is a sociocultural anthropologist researching how studies of the human microbiome intersect with biomedical ethics, pu

In [37]:
hass_faculty_data = pd.DataFrame(all_faculty_json)


In [38]:
hass_faculty_data.head()

,Name,Title,Contact Info,Profile URL,Sections
0,Amber Benezra,Academics,"{'Address': 'Peirce 209', 'Phone': '(201) 216-...",https://www.stevens.edu/profile/abenezra,{'Education': 'PhD (2014) New School for Socia...
1,Michelle Burke,Academics,"{'Address': 'Kidde 226', 'Phone': '(201) 216-3...",https://www.stevens.edu/profile/mburke3,{'Education': 'PhD (2014) University of Cincin...
2,Diana Bush,Academics,"{'Address': 'Morton 206', 'Phone': '(201) 216-...",https://www.stevens.edu/profile/dbush,{'Research': 'Primary fields: Modern and conte...
3,Sean Cashbaugh,Academics,"{'Address': '', 'Phone': '(201) 216-8502', 'Em...",https://www.stevens.edu/profile/scashbau,{'Education': 'PhD (2016) University of Texas ...
4,Betul Cihan Artun,Academics,"{'Address': 'Peirce 107', 'Phone': '(201) 216-...",https://www.stevens.edu/profile/fcihanar,{'Education': 'PhD (2016) University of Massac...


In [39]:
hass_faculty_data=hass_faculty_data.drop_duplicates(subset=["Name"])

In [40]:
hass_faculty_data.to_csv("./ProfessorData/HASS_school/faculty_directory.csv")
hass_faculty_data.to_json("./ProfessorData/HASS_school/faculty_directory.json")